In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import classification_report, f1_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from kneed import KneeLocator
from sklearn.model_selection import cross_validate
from sklearn.utils import resample

In [5]:
#Configuración de RF
def training_RF(dataset):
    ws_, ov_ = map(int, dataset.split("_"))
    print(f"Training Random Forest for dataset: {dataset}")
    
    data = pd.read_csv(f"../data/preprocessed_data/Mhealth_{dataset}.csv")
    
    data.drop(["Sensor", "ID", "DTS", "Subject"], axis=1, inplace=True)
    
    Y_tChest = data["Activity"]
    Y_tLA = data["Activity"]
    Y_tRLA = data["Activity"]
    
    data.drop("Activity", axis=1, inplace=True)
    
    X = data[data.columns.to_list()]
    
    X_tChest = X.filter(like='tChest')
    X_tLA = X.filter(like='tLA')
    X_tRLA = X.filter(like='tRLA')
    
    x_tChest, X_test_tChest, y_tChest , y_test_tChest = train_test_split(X_tChest,Y_tChest,test_size=0.3, stratify=Y_tChest, random_state=0)
    x_tLA, X_test_tLA, y_tLA , y_test_tLA = train_test_split(X_tLA,Y_tLA,test_size=0.3, stratify=Y_tLA, random_state=0)
    x_tRLA, X_test_tRLA, y_tRLA , y_test_tRLA = train_test_split(X_tRLA,Y_tRLA,test_size=0.3, stratify=Y_tRLA, random_state=0)
    
    #Training Random Forest
    print("Training Random Forest")
    max_depth = 15
    n_repeats = 30
    k_folds = 10
    
    rf = np.zeros(max_depth)
    rf_wf1_tChest = np.zeros(max_depth)
    rf_wf1_tLA = np.zeros(max_depth)
    rf_wf1_tRLA = np.zeros(max_depth)
    
    count = 1
    
    for i in range (1, max_depth + 1):
        rf[count-1] = i
        
        print(f"RF {dataset}, max_depth:",i)
    
        cv = RepeatedStratifiedKFold(n_splits=k_folds, n_repeats=n_repeats, random_state=0)
        
        model = RandomForestClassifier(max_depth=i, random_state=0)
        
        scoring = ['f1_weighted']
        
        scores_tChest = cross_validate(model, x_tChest, y_tChest, scoring=scoring, cv=cv, n_jobs=-1)
        scores_tLA = cross_validate(model, x_tLA, y_tLA, scoring=scoring, cv=cv, n_jobs=-1)
        scores_tRLA = cross_validate(model, x_tRLA, y_tRLA, scoring=scoring, cv=cv, n_jobs=-1)
        
        rf_wf1_tChest[count - 1] = scores_tChest['test_f1_weighted'].mean()
        rf_wf1_tLA[count - 1] = scores_tLA['test_f1_weighted'].mean()
        rf_wf1_tRLA[count - 1] = scores_tRLA['test_f1_weighted'].mean()
        
        count = count + 1
    
    wf1_df = pd.DataFrame({
        'dataset': [dataset] * max_depth,
        'max_depth': rf,
        'weighted_f1_tChest': rf_wf1_tChest,
        'weighted_f1_tLA': rf_wf1_tLA,
        'weighted_f1_tRLA': rf_wf1_tRLA
    })
    ##Curva de resultados con distintos max_depth
    wf1_df.to_csv(f"../data/outputs/training_results_{dataset}.csv", index=False)
    
    #Selección de max_depth
    print("Selecting max_depth")
    tChest_kneedle = KneeLocator(rf.tolist(), rf_wf1_tChest.tolist(), S=1.0, curve="concave", direction="increasing")
    tChest_knee = int(round(tChest_kneedle.knee))
    
    tLA_kneedle = KneeLocator(rf.tolist(), rf_wf1_tLA.tolist(), S=1.0, curve="concave", direction="increasing")
    tLA_knee = int(round(tLA_kneedle.knee))
    
    tRLA_kneedle = KneeLocator(rf.tolist(), rf_wf1_tRLA.tolist(), S=1.0, curve="concave", direction="increasing")
    tRLA_knee = int(round(tRLA_kneedle.knee))
    
    tChest_model = RandomForestClassifier(max_depth=tChest_knee, random_state=0)
    tChest_model.fit(x_tChest, y_tChest)  
    y_test_hat = tChest_model.predict(X_test_tChest)
    tChest_report = classification_report(y_test_tChest, y_test_hat, output_dict=True)
    tChest_wf1 = tChest_report['weighted avg']['f1-score']
    
    tLA_model = RandomForestClassifier(max_depth=tLA_knee, random_state=0)
    tLA_model.fit(x_tLA, y_tLA)  
    y_test_hat = tLA_model.predict(X_test_tLA)
    tLA_report = classification_report(y_test_tLA, y_test_hat, output_dict=True)
    tLA_wf1 = tLA_report['weighted avg']['f1-score']
    
    tRLA_model = RandomForestClassifier(max_depth=tRLA_knee, random_state=0)
    tRLA_model.fit(x_tRLA, y_tRLA)  
    y_test_hat = tRLA_model.predict(X_test_tRLA)
    tRLA_report = classification_report(y_test_tRLA, y_test_hat, output_dict=True)
    tRLA_wf1 = tRLA_report['weighted avg']['f1-score']
    
    wf1_values = [tChest_wf1, tLA_wf1, tRLA_wf1]
    max_depth_values = [tChest_knee, tLA_knee, tRLA_knee]
    name = [f'tChest_{dataset}', f'tLA_{dataset}', f'tRLA_{dataset}']
    
    test_results_df = pd.DataFrame({
    'wf1': wf1_values,
    'max_depth': max_depth_values,
    'dataset': name  
    })
    #WF1 con max_depth optimo
    test_results_df.to_csv(f"../data/outputs/test_results_{dataset}.csv", index=False)

In [6]:
#Generación de los modelos
training_RF("50_25")
training_RF("50_50")
training_RF("50_75")
training_RF("100_25")
training_RF("100_50")
training_RF("100_75")
training_RF("150_25")
training_RF("150_50")
training_RF("150_75")

Training Random Forest
RF 50_25, max_depth: 1
RF 50_25, max_depth: 2
RF 50_25, max_depth: 3
RF 50_25, max_depth: 4
RF 50_25, max_depth: 5
RF 50_25, max_depth: 6
RF 50_25, max_depth: 7
RF 50_25, max_depth: 8
RF 50_25, max_depth: 9
RF 50_25, max_depth: 10
RF 50_25, max_depth: 11
RF 50_25, max_depth: 12
RF 50_25, max_depth: 13
RF 50_25, max_depth: 14
RF 50_25, max_depth: 15
Selecting max_depth
Training final model to 50_25
Training Random Forest
RF 50_50, max_depth: 1
RF 50_50, max_depth: 2
RF 50_50, max_depth: 3
RF 50_50, max_depth: 4
RF 50_50, max_depth: 5
RF 50_50, max_depth: 6
RF 50_50, max_depth: 7
RF 50_50, max_depth: 8
RF 50_50, max_depth: 9
RF 50_50, max_depth: 10
RF 50_50, max_depth: 11
RF 50_50, max_depth: 12
RF 50_50, max_depth: 13
RF 50_50, max_depth: 14
RF 50_50, max_depth: 15
Selecting max_depth
Training final model to 50_50
Training Random Forest
RF 50_75, max_depth: 1
RF 50_75, max_depth: 2
RF 50_75, max_depth: 3
RF 50_75, max_depth: 4
RF 50_75, max_depth: 5
RF 50_75, max_

/Users/mario.quinde/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/mario.quinde/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/mario.quinde/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Training final model to 150_25
Training Random Forest
RF 150_50, max_depth: 1
RF 150_50, max_depth: 2
RF 150_50, max_depth: 3
RF 150_50, max_depth: 4
RF 150_50, max_depth: 5
RF 150_50, max_depth: 6
RF 150_50, max_depth: 7
RF 150_50, max_depth: 8
RF 150_50, max_depth: 9
RF 150_50, max_depth: 10
RF 150_50, max_depth: 11
RF 150_50, max_depth: 12
RF 150_50, max_depth: 13
RF 150_50, max_depth: 14
RF 150_50, max_depth: 15
Selecting max_depth
Training final model to 150_50
Training Random Forest
RF 150_75, max_depth: 1
RF 150_75, max_depth: 2
RF 150_75, max_depth: 3
RF 150_75, max_depth: 4
RF 150_75, max_depth: 5
RF 150_75, max_depth: 6
RF 150_75, max_depth: 7
RF 150_75, max_depth: 8
RF 150_75, max_depth: 9
RF 150_75, max_depth: 10
RF 150_75, max_depth: 11
RF 150_75, max_depth: 12
RF 150_75, max_depth: 13
RF 150_75, max_depth: 14
RF 150_75, max_depth: 15
Selecting max_depth
Training final model to 150_75


In [15]:
# Gráficas de resultados RF (versión optimizada)
import pandas as pd
import matplotlib.pyplot as plt
from kneed import KneeLocator
import os

def plot_rf_results(dataset):
    data= pd.read_csv(f"../data/outputs/training_results_{dataset}.csv")
    metrics = ["weighted_f1_tChest", "weighted_f1_tLA", "weighted_f1_tRLA"]
    names = ["tChest", "tLA", "tRLA"]

    for i, metric in enumerate(metrics):
        m_d = data["max_depth"]
        wf1 = data[metric]

        kneedle = KneeLocator(m_d, wf1, S=1.0, curve="concave", direction="increasing")

        # Gráfica normalizada
        plt.figure()
        kneedle.plot_knee_normalized()
        plt.title(f"Normalized {dataset} - {names[i]}")
        plt.tight_layout()

        filename = f"normalized_{dataset}_{names[i]}.png"
        filepath = os.path.join("../data/outputs/imgs", filename)
        plt.savefig(filepath, dpi=300)
        plt.close()
        plt.close()


In [16]:
plot_rf_results('50_25')
plot_rf_results('50_50')
plot_rf_results('50_75')
plot_rf_results('100_25')
plot_rf_results('100_50')
plot_rf_results('100_75')
plot_rf_results('150_25')
plot_rf_results('150_50')
plot_rf_results('150_75')

In [16]:
def shape_per_dataset(dataset):
    df = pd.read_csv(f"../data/preprocessed_data/Mhealth_{dataset}.csv")
    conteo = df['Subject'].value_counts()
    #print(df.columns)
    print(f"Dataset: {dataset}")
    print("Shape of the dataset:")
    print(df.shape)
    #print("Number of samples per subject:")
    #print(conteo)
    
shape_per_dataset('50_25')
shape_per_dataset('50_50')
shape_per_dataset('50_75')
shape_per_dataset('100_25')
shape_per_dataset('100_50')
shape_per_dataset('100_75')
shape_per_dataset('150_25')
shape_per_dataset('150_50')
shape_per_dataset('150_75')

Dataset: 50_25
Shape of the dataset:
(8931, 194)
Dataset: 50_50
Shape of the dataset:
(13512, 194)
Dataset: 50_75
Shape of the dataset:
(25997, 194)
Dataset: 100_25
Shape of the dataset:
(4464, 194)
Dataset: 100_50
Shape of the dataset:
(6692, 194)
Dataset: 100_75
Shape of the dataset:
(13272, 194)
Dataset: 150_25
Shape of the dataset:
(2911, 194)
Dataset: 150_50
Shape of the dataset:
(4358, 194)
Dataset: 150_75
Shape of the dataset:
(8579, 194)


In [13]:
df = pd.read_csv(
    "D:/Estudios/Tesis/CODIGO sin modificar/CODIGO/windows_datasets_FJSMC/windows_datasets_FJSMC/datasets/data/CON_INDICE_NOMBRES/MHEALTHDATASET_INDICE_letras.csv",
    sep=";"  # este es el detalle importante
)
print(df.columns)
conteo = df['Subject'].value_counts()
print("Shape of the dataset:")
print(df.shape)
print("Number of samples per subject:")
print(conteo)



Index(['ID_MHE', 'Subject', 'Activity_Number', 'Sensor',
       'Acctr_tChest_X(m/s2)', 'Acctr_tChest_Y(m/s2)', 'Acctr_tChest_Z(m/s2)',
       'L1_tChest(Mv)', 'L2_tChest(Mv)', 'Acctr_tLA_X(m/s2)',
       'Acctr_tLA_Y(m/s2)', 'Acctr_tLA_Z(m/s2)', 'Gype_tLA_X(deg/s)',
       'Gype_tLA_Y(deg/s)', 'Gype_tLA_Z(deg/s)', 'MgFld_tLA_X(uT)',
       'MgFld_tLA_X(T)', 'MgFld_tLA_Y(uT)', 'MgFld_tLA_Y(T)',
       'MgFld_tLA_Z(uT)', 'MgFld_tLA_Z(T)', 'Acctr_tRLA_X(m/s2)',
       'Acctr_tRLA_Y(m/s2)', 'Acctr_tRLA_Z(m/s2)', 'Gype_tRLA_X(deg/s)',
       'Gype_tRLA_Y(deg/s)', 'Gype_tRLA_Z(deg/s)', 'MgFld_tRLA_X(uT)',
       'MgFld_tRLA_X(T)', 'MgFld_tRLA_Y(uT)', 'MgFld_tRLA_Y(T)',
       'MgFld_tRLA_Z(uT)', 'MgFld_tRLA_Z(T)'],
      dtype='object')
Shape of the dataset:
(1215745, 33)
Number of samples per subject:
Subject
MHE_S_01    161280
MHE_S_09    135168
MHE_S_02    130561
MHE_S_08    129024
MHE_S_03    122112
MHE_S_05    119808
MHE_S_04    116736
MHE_S_07    104448
MHE_S_06     98304
MHE_S_10    